# CompactFactExtraction — проверка релиза Qwen на одной T4
Загрузите **Qwen_release_Colab.zip**. Модель уже обучена: этот notebook проверяет упаковку, качество, скорость и устойчивость. `date_extractor` и каскада нет. Все прогоны вызываются прямо в ячейках, с выводом времени **каждого звонка**. Переобучение не запускается.

В Colab выберите **Runtime → Change runtime type → T4 GPU**, затем выполните ячейки по порядку. База скачивается один раз (~1 GB); выбранный адаптер уже в ZIP. Контрольный полный проход dev+val и дополнительные проверки могут занять около 1,5–2 часов на T4; это оценка по прежним 5,7 с/звонок, а не гарантия времени.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, json, os, sys, hashlib
uploaded=files.upload()
assert 'Qwen_release_Colab.zip' in uploaded, 'Нужен Qwen_release_Colab.zip'
ROOT=Path('/content/qwen_release');ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile('Qwen_release_Colab.zip') as z:
    for m in z.infolist():
        assert (ROOT/m.filename).resolve().is_relative_to(ROOT.resolve()),m.filename
    z.extractall(ROOT)
del uploaded
manifest=json.loads((ROOT/'PACKAGE_SHA256.json').read_text())
for name,expected in manifest.items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest()==expected,name
os.chdir(ROOT);sys.path.insert(0,str(ROOT))
print('Архив проверен:',len(manifest),'файлов')


## Зависимости
Запускайте в свежей сессии. Для сопоставимости с Docker фиксируем Torch 2.8.0 (CUDA 12.8). Первичная установка может быть долгой. Предустановленные torchvision/torchaudio могут быть несовместимы с этим Torch. Для текстового Qwen эти пакеты и librosa не нужны: удаляем их, чтобы Transformers не импортировал лишние vision/audio зависимости. После установки обязательно перезапустите сессию, как указано ниже.


In [ ]:
%pip uninstall -y torchvision torchaudio librosa
%pip install -q torch==2.8.0 --index-url https://download.pytorch.org/whl/cu128
%pip install -q -r requirements.lock matplotlib==3.10.3


## Обязательно перезапустите сессию
После установки выберите **Runtime → Restart session** (не Disconnect and delete runtime). Затем продолжайте со следующей ячейки. Повторно загружать ZIP и устанавливать пакеты не нужно: файлы в /content сохраняются. Перезапуск очищает уже загруженные несовместимые модули.


In [ ]:
from pathlib import Path
import os, sys, json, zipfile
ROOT=Path('/content/qwen_release')
assert ROOT.exists(), 'Папка исчезла: повторите загрузку архива'
os.chdir(ROOT)
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from google.colab import files
import importlib.util
assert importlib.util.find_spec('torchvision') is None, 'Выполните ячейку удаления необязательных пакетов и перезапустите сессию'
import torch, transformers, peft, unittest, gc
from transformers import PreTrainedModel, Qwen2ForCausalLM
print('Импорт текстовой модели Qwen: OK')
assert torch.cuda.is_available(), 'GPU не доступна — выберите T4'
assert torch.__version__.split('+')[0]=='2.8.0', 'Перезапустите runtime после установки Torch'
print('GPU:',torch.cuda.get_device_name(0),'Torch:',torch.__version__,'Transformers:',transformers.__version__,'PEFT:',peft.__version__)
suite=unittest.defaultTestLoader.discover(str(ROOT/'tests'))
assert unittest.TextTestRunner(verbosity=2).run(suite).wasSuccessful()
from tools.verify_raw_codec import main as verify_codec
verify_codec()


## Веса и автономная модель
Скачивание закреплённой ревизии базы; merge в FP16 на CPU, как в локальной поставке. Дальнейший инференс использует только локальные файлы. Выбор checkpoint уже сделан на dev — здесь ничего не переобучаем и не выбираем по val.


In [ ]:
import tools.fetch_base
from tools.merge_model import main as merge_model
merge_model()
os.environ['HF_HUB_OFFLINE']='1';os.environ['TRANSFORMERS_OFFLINE']='1'
print((ROOT/'models/merged/release_manifest.json').read_text())
gc.collect();torch.cuda.empty_cache()


## Куда сохранять результаты
Рекомендуется Google Drive: предсказания и время каждого звонка записываются сразу. После перерыва повторите подготовительные ячейки; `--resume` продолжит пропущенные звонки с теми же настройками. При разрыве посреди записи проверка остановится с ошибкой — используйте новую папку RUN, не редактируйте результаты вручную.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN=Path('/content/drive/MyDrive/CFE_Qwen_release_check_v1')
RUN.mkdir(parents=True,exist_ok=True)
# CPU-замеры были выполнены локально; не подменяем их CPU Colab.
import shutil
for p in (ROOT/'benchmarks').glob('curve24_cpu.jsonl*'):
    if not (RUN/p.name).exists():shutil.copy2(p,RUN/p.name)
from src.cli import main as infer
from tools.score import score

def run(name,input_name,context='full'):
    print('\n▶',name,flush=True)
    output=RUN/(name+'.jsonl')
    infer(['--input',str(ROOT/'data'/input_name),'--output',str(output),
           '--device','cuda','--dtype','float16','--context',context,
           '--threads','4','--warmup','--resume'])
    gc.collect();torch.cuda.empty_cache()
    return output
print('Результаты:',RUN)


## 1. Основной режим: полный dev, 385 звонков
Прогресс ниже — по каждому звонку. Время загрузки и прогрева пишется отдельно, в latency звонков не входит. Предсказания, сырые ответы, счётчики ошибок и индивидуальные времена сохраняются в RUN.


In [ ]:
dev_output=run('dev_gpu_full','dev_input.jsonl')
score(ROOT/'data/dev_gold.jsonl',dev_output,RUN/'dev_metrics.json')


## 2. Третья точка ресурсы × качество: только целевые реплики
24 звонка заранее выбраны по длине текста, без доступа к разметке. Сравнение: локальный CPU FP32 полный контекст; T4 FP16 полный контекст; T4 FP16 только целевые реплики. Уменьшенный контекст — исследовательский режим, он **не заменяет** основной режим релиза автоматически.


In [ ]:
run('curve24_gpu_target','curve24_input.jsonl',context='target_only')


## 3. Усиление шума и ловушки
Два уровня фонетических искажений и слов-паразитов на одних и тех же 96 dev-звонках. Даты, числа и отрицания сохранены. Это ограниченный собственный stress-test, а не скрытый adversarial организаторов. Дополнительно пять вручную составленных звонков, в которых подтверждённых фактов нет.


In [ ]:
run('stress96_mild_gpu','stress96_mild_input.jsonl')
run('stress96_strong_gpu','stress96_strong_input.jsonl')
traps=run('traps_gpu','traps_input.jsonl')
from tools.check_traps import check
check(traps,RUN/'traps_check.json')


## 4. Официальный val: только оценка замороженного основного режима
Здесь выполняется повторная генерация после упаковки. По её результату не выбирается контекст, модель или правила. Сравнение со старым JSONL покажет, изменилась ли генерация при переносе версии Torch и merge.


In [ ]:
val_output=run('val_gpu_full','val_input.jsonl')
score(ROOT/'data/val_gold.jsonl',val_output,RUN/'val_metrics.json')


## 5. Настоящий HTTP-запрос к тому же сервису
Проверяем три звонка на локальном HTTP-порту Colab. Сервер работает в Python-потоке только на время проверки; генерация вызывается синхронно, статус каждого запроса печатается. Это проверка Linux/GPU API, **не Docker-тест**: Docker Engine в обычном Colab недоступен.


In [ ]:
import threading,time,uvicorn
from src.extractor import Extractor
from src.api import create_app
from tools.benchmark_http import main as benchmark_http
engine=Extractor(device='cuda',dtype='float16')
server=uvicorn.Server(uvicorn.Config(create_app(lambda:engine),host='127.0.0.1',port=8765,log_level='warning'))
thread=threading.Thread(target=server.run,daemon=True);thread.start()
try:
    for _ in range(100):
        if server.started:break
        time.sleep(.1)
    assert server.started,'HTTP server failed to start'
    sample=ROOT/'data/http3_input.jsonl'
    sample.write_text('\n'.join((ROOT/'data/curve24_input.jsonl').read_text().splitlines()[:3])+'\n')
    benchmark_http(['--url','http://127.0.0.1:8765','--input',str(sample),'--output',str(RUN/'http_gpu.jsonl')])
finally:
    server.should_exit=True;thread.join(timeout=30)
    del engine,server;gc.collect();torch.cuda.empty_cache()


## 6. Таблица, график и результат
Если CPU-результатов в архиве нет, отчёт честно укажет неполную кривую. Не добавляйте приблизительные точки вручную. Пришлите **Qwen_release_results.zip** — по нему завершим отчёт и проверим регрессию. Это архив измерений, не финальная сдача жюри.


In [ ]:
from tools.summarize_run import main as summarize
summarize(RUN)
from IPython.display import display,Markdown,Image
display(Markdown((RUN/'reports/RESULTS.md').read_text()))
if (RUN/'reports/resource_quality.png').exists():display(Image(filename=str(RUN/'reports/resource_quality.png')))
result=Path('/content/Qwen_release_results.zip')
with zipfile.ZipFile(result,'w',zipfile.ZIP_DEFLATED) as z:
    for p in RUN.rglob('*'):
        if p.is_file():z.write(p,str(p.relative_to(RUN)))
    z.write(ROOT/'models/merged/release_manifest.json','merged_release_manifest.json')
    z.write(ROOT/'PACKAGE_SHA256.json','input_package_sha256.json')
print('Готово:',result,round(result.stat().st_size/1024**2,1),'MiB')
files.download(str(result))
